# Neighborhood-MIL baseline — single dataset walkthrough

How to run the **local-neighborhood multiple-instance-learning (MIL)** baseline for one
dataset, alongside the global composition baseline it parallels.

The idea, in MIL terms:

| MIL concept | here |
|---|---|
| **bag** | a region (TMA core / ROI) — the thing we have a label for |
| **instance** | a **local neighborhood** (cells within `radius_um` of a focal cell), via `RegionData.neighborhood` → `RegionData.subset` |
| **instance features** | any existing region featurizer applied to the neighborhood (a 100 µm patch is described just like a whole region) |
| **bag → instance labels** | each sampled neighborhood inherits its region's label (standard MIL assumption) |
| **instance model** | any existing region model — `GradientBoostingModel` (the XGBoost-equivalent), `RandomForestModel`, `LinearClassifier` |
| **bag prediction** | sample a region's neighborhoods, score each, **pool** the probabilities (mean) |

`NeighborhoodMILModel` is a **thin wrapper**: you hand it a featurizer instance and a model
instance, and it only does the MIL plumbing (sample → label → pool), delegating the actual
learning to the model you passed.

| piece | class | module |
|---|---|---|
| Instance featurizer | `CompositionFeaturizer` / `MeanExpressionFeaturizer` | `benchmark.features.basic_feats` |
| Instance model | `GradientBoostingModel` / `RandomForestModel` / `LinearClassifier` | `benchmark.models` |
| MIL wrapper | `NeighborhoodMILModel` | `benchmark.models.mil` |

We use **HNC-Wu2022** (CODEX) — same dataset as the composition walkthrough, so the numbers
are directly comparable.

## 0. Setup — put the `benchmark` package on the path

In [ ]:
import sys
import numpy as np
import pandas as pd

data_root = Path(".").resolve().parent.parent
code_root = Path(".").resolve().parent
sys.path.insert(0, code_root)

from benchmark.utils.registry import load_dataset

## 1. Load the dataset and one CV fold

Same plumbing as the composition baseline: patient-level K-fold within the CV cohort.

In [2]:
from benchmark.validation.splits import safe_patient_kfold, stratify_column

ds = load_dataset("hnc_wu2022", data_root=data_root)
ct_col = ds.config.get("cell_type_col", "cell_type")
vcfg = ds.validation_config

task = "primary_outcome"
task_cfg = ds.get_task_config(task)

meta = ds.get_task_metadata(task)
cv_meta = meta.query(vcfg["cv_filter"]).reset_index(drop=True)
folds = safe_patient_kfold(cv_meta, n_folds=5, patient_col="patient_id",
                           stratify_col=stratify_column(task_cfg), seed=0)
train_ids, val_ids = folds[0]
print("Dataset      :", ds)
print("Task         :", task, f"({task_cfg['type']})")
print("cell_type_col:", ct_col)
print(f"fold 0: {len(train_ids)} train regions, {len(val_ids)} val regions")

Dataset      : TMEDataset(name='HNC-Wu2022', root=/Users/zhenqin/WORK/TME_benchmark/HNC_Wu2022/processed)
Task         : primary_outcome (binary_classification)
cell_type_col: cell_type
fold 0: 243 train regions, 65 val regions


## 2. What an "instance" is — sample one neighborhood and featurize it

The MIL model takes the **`RegionData` objects directly** (not a precomputed feature table)
because it has to carve neighborhoods out of them itself. Each neighborhood is a `RegionData`,
so *any* region featurizer works on it unchanged — here, cell-type composition of the 100 µm
patch. (The wrapper fits the featurizer's vocabulary on the train regions internally; we fit
one here just to peek at a single instance.)

In [3]:
from benchmark.features.basic_feats import CompositionFeaturizer

train_regions = ds.load_regions(train_ids)
val_regions   = ds.load_regions(val_ids)

peek = CompositionFeaturizer(cell_type_col=ct_col).fit(train_regions)

r0 = train_regions[0]
center = r0.coordinates.index[r0.n_cells // 2]
nb = r0.neighborhood(center, radius_um=100.0)
print(f"region {r0.region_id}: {r0.n_cells} cells")
print(f"100µm neighborhood around cell {center}: {nb.n_cells} cells")
pd.Series(peek.extract_region(nb), name="instance_features").round(3)

region UPMC_c004_v001_r001_reg063: 2584 cells
100µm neighborhood around cell 938965: 161 cells


APC                     0.019
B cell                  0.056
CD4 T cell              0.099
CD8 T cell              0.130
Granulocyte             0.019
Lymph vessel            0.006
Macrophage              0.062
Naive immune cell       0.025
Stromal / Fibroblast    0.025
Tumor                   0.050
Tumor (CD15+)           0.012
Tumor (CD20+)           0.000
Tumor (CD21+)           0.025
Tumor (Ki67+)           0.186
Tumor (Podo+)           0.186
Unassigned              0.031
Vessel                  0.068
Name: instance_features, dtype: float64

## 3. Fit the MIL model on one fold

You only specify four things: a **featurizer instance**, a **model instance**, how many
neighborhoods to sample per region (`n_samples`, or `sample_ratio`), and the `radius_um`.

`fit(regions, target)` fits the featurizer's vocabulary on the train regions, samples
`n_samples` neighborhoods per region, labels each with its region's label, and calls the
wrapped `model.fit`. `predict(regions)` re-samples neighborhoods, calls `model.predict`, and
mean-pools the probabilities back to one row per region — so it returns the same
`(n_regions, n_classes)` matrix the metric scorer expects.

In [5]:
from benchmark.models.boosting import GradientBoostingModel
from benchmark.models.mil import NeighborhoodMILModel
from benchmark.validation.metrics import score_predictions

y_tr = ds.build_target(train_ids, task)
y_va = ds.build_target(val_ids, task)

feat = CompositionFeaturizer(cell_type_col=ct_col).fit(train_regions)

model = NeighborhoodMILModel(
    featurizer=feat,
    model=GradientBoostingModel(seed=0),
    n_samples=0.05,
    radius_um=100.0,
).fit(train_regions, y_tr)

proba_tr = model.predict(train_regions, n_samples=0.01)
proba_va = model.predict(val_regions, n_samples=1)

print("classes_:", model.classes_, "| pred shape:", proba_va.shape)
print(f"\nTask: {task}  (neighborhood-MIL, composition @100µm, gradient boosting)")
print(f"trained on {len(train_ids)} regions:", score_predictions(task_cfg, y_tr, proba_tr, model.classes_))
print(f"validated on {len(val_ids)} regions:", score_predictions(task_cfg, y_va, proba_va, model.classes_))

classes_: [0 1] | pred shape: (65, 2)

Task: primary_outcome  (neighborhood-MIL, composition @100µm, gradient boosting)
trained on 243 regions: {'auc_roc': 0.9997037475929492, 'avg_precision': 0.9998400036512638, 'balanced_acc': 0.9840764331210191}
validated on 65 regions: {'auc_roc': 0.665, 'avg_precision': 0.7782292477593306, 'balanced_acc': 0.63}


## 4. Swapping the pieces

Because the model is just `featurizer + model` instances, trying variants is a one-liner each
— swap the featurizer instance, swap the model instance, or change the radius / sampling:

- **mean-expression** instances instead of composition (marker profile of each patch);
- **random forest** / **logistic regression** instead of gradient boosting;
- a tighter **50 µm** radius;
- **`sample_ratio`** instead of a fixed `n_samples`.

In [6]:
from benchmark.features.basic_feats import MeanExpressionFeaturizer
from benchmark.models import RandomForestModel, LinearClassifier

comp_feat = CompositionFeaturizer(cell_type_col=ct_col).fit(train_regions)
expr_feat = MeanExpressionFeaturizer().fit(train_regions)

variants = {
    "composition + boosting": NeighborhoodMILModel(
        comp_feat, GradientBoostingModel(seed=0),
        n_samples=24, radius_um=100.0),
    "mean-expr + boosting": NeighborhoodMILModel(
        expr_feat, GradientBoostingModel(seed=0),
        n_samples=24, radius_um=100.0),
    "composition + forest": NeighborhoodMILModel(
        comp_feat, RandomForestModel(seed=0),
        n_samples=24, radius_um=100.0),
    "composition + linear": NeighborhoodMILModel(
        comp_feat, LinearClassifier(seed=0),
        n_samples=24, radius_um=100.0),
    "composition + boosting (50µm)": NeighborhoodMILModel(
        comp_feat, GradientBoostingModel(seed=0),
        n_samples=24, radius_um=50.0),
    "composition + boosting (ratio=1%)": NeighborhoodMILModel(
        comp_feat, GradientBoostingModel(seed=0),
        n_samples=0.01, radius_um=100.0),
}

rows = []
for name, m in variants.items():
    m.fit(train_regions, y_tr)
    s = score_predictions(task_cfg, y_va, m.predict(val_regions), m.classes_)
    rows.append({"variant": name, **{k: round(v, 3) for k, v in s.items()}})
pd.DataFrame(rows).set_index("variant")

,auc_roc,avg_precision,balanced_acc
variant,,,
composition + boosting,0.718,0.805,0.653
mean-expr + boosting,0.543,0.744,0.440
composition + forest,0.707,0.793,0.672
composition + linear,0.644,0.759,0.600
composition + boosting (50µm),0.703,0.766,0.657
composition + boosting (ratio=1%),0.671,0.790,0.625


## 5. Full cross-validation across seeds

The model consumes `RegionData` rather than a feature table, so we run the patient-level CV
loop directly (rather than through `cross_validate`, which assumes the featurize-then-fit
split). One mean ± SD over folds × seeds, exactly like the other baselines report. A fresh
model instance is built per fold (each `fit` re-fits both the featurizer and the wrapped
model).

In [9]:
from benchmark.validation.metrics import PRIMARY_METRIC, summarize_folds

def make_mil(seed):
    return NeighborhoodMILModel(
        featurizer=CompositionFeaturizer(cell_type_col=ct_col),
        model=GradientBoostingModel(seed=seed),
        n_samples=24, radius_um=100.0, seed=seed)

fold_metrics = []
for seed in [0, 1, 2]:
    folds = safe_patient_kfold(cv_meta, n_folds=5, patient_col="patient_id",
                               stratify_col=stratify_column(task_cfg), seed=seed)
    for fold_i, (tr_ids, va_ids) in enumerate(folds):
        print(f"seed {seed} fold {fold_i}: {len(tr_ids)} train regions, {len(va_ids)} val regions")
        tr_r, va_r = ds.load_regions(tr_ids), ds.load_regions(va_ids)
        y_t, y_v = ds.build_target(tr_ids, task), ds.build_target(va_ids, task)
        try:
            comp_feat = CompositionFeaturizer(cell_type_col=ct_col).fit(tr_r)
            model = NeighborhoodMILModel(
                featurizer=comp_feat,
                model=GradientBoostingModel(seed=seed),
                n_samples=0.1, radius_um=100.0, seed=seed).fit(tr_r, y_t)
            metrics = score_predictions(task_cfg, y_v, model.predict(va_r), model.classes_)
        except Exception as e:
            metrics = {"auc_roc": float("nan")}
        metrics.update(seed=seed, fold=fold_i, n_train=len(tr_ids), n_val=len(va_ids))
        fold_metrics.append(metrics)

fm = pd.DataFrame(fold_metrics)
metric = PRIMARY_METRIC[task_cfg["type"]]
mean, sd = summarize_folds(fold_metrics, metric)
print(f"neighborhood-MIL CV {metric}: {mean:.4f} +/- {sd:.4f}  (3 seeds x 5 folds)")
fm.round(3)

seed 0 fold 0: 243 train regions, 65 val regions
seed 0 fold 1: 240 train regions, 68 val regions
seed 0 fold 2: 245 train regions, 63 val regions
seed 0 fold 3: 248 train regions, 60 val regions
seed 0 fold 4: 256 train regions, 52 val regions
seed 1 fold 0: 246 train regions, 62 val regions
seed 1 fold 1: 244 train regions, 64 val regions
seed 1 fold 2: 250 train regions, 58 val regions
seed 1 fold 3: 248 train regions, 60 val regions
seed 1 fold 4: 244 train regions, 64 val regions
seed 2 fold 0: 246 train regions, 62 val regions
seed 2 fold 1: 247 train regions, 61 val regions
seed 2 fold 2: 252 train regions, 56 val regions
seed 2 fold 3: 245 train regions, 63 val regions
seed 2 fold 4: 242 train regions, 66 val regions
neighborhood-MIL CV auc_roc: 0.7724 +/- 0.0977  (3 seeds x 5 folds)


,auc_roc,avg_precision,balanced_acc,seed,fold,n_train,n_val
0,0.667,0.784,0.630,0,0,243,65
1,0.766,0.871,0.682,0,1,240,68
2,0.902,0.954,0.807,0,2,245,63
3,0.742,0.879,0.662,0,3,248,60
4,0.821,0.921,0.743,0,4,256,52
5,0.833,0.925,0.718,1,0,246,62
6,0.827,0.896,0.712,1,1,244,64
7,0.724,0.816,0.693,1,2,250,58
8,0.701,0.861,0.625,1,3,248,60
9,0.819,0.930,0.771,1,4,244,64


## Notes

- **Thin wrapper, reused parts.** `NeighborhoodMILModel` adds only the MIL plumbing
  (`neighborhood`/`subset` sampling, bag→instance labelling, probability pooling). The
  featurizer, the instance classifier (`GradientBoostingModel` / `RandomForestModel` /
  `LinearClassifier`, all sharing `_TabularModel`'s impute+scale), the splits and the metrics
  are the same shared components the global baselines use.
- **`GradientBoostingModel`** is `HistGradientBoostingClassifier` — the dependency-free
  XGBoost-equivalent — and lives in `benchmark.models` next to `RandomForestModel`, so it is
  also usable on its own as a global baseline.
- **Sampling.** `n_samples` is a fixed count per region; `sample_ratio` scales with region
  size (`ceil(ratio * n_cells)`). The same setting is used at fit and predict.
- **Survival** is out of scope here (instance-level Cox needs a different pooling story); MIL
  covers the binary / multiclass classification tasks.